# SQL Planning and Safety

This notebook builds the controlled SQL layer of the AI Analytics Assistant.

An approved business question will first be converted into a structured SQL plan. The generated SQL will then pass deterministic safety checks, PostgreSQL preflight checks and read-only execution controls before it is allowed to query the database.

## Section 1 - Structured SQL Plan

This section defines the structured plan that sits between question analysis and SQL generation.

Instead of asking the LLM to immediately write SQL, the system first records the metric, tables, joins, filters, grouping, ordering and result limit it intends to use. This makes the intended query easier to inspect and validate before any SQL is produced.

In [1]:
from enum import Enum

from pydantic import (
    BaseModel,
    ConfigDict,
    model_validator
)

In [2]:
class JoinType(str, Enum):
    INNER = "INNER"
    LEFT = "LEFT"


class FilterOperator(str, Enum):
    EQ = "EQ"
    NE = "NE"
    GT = "GT"
    GTE = "GTE"
    LT = "LT"
    LTE = "LTE"
    IN = "IN"
    NOT_IN = "NOT_IN"
    BETWEEN = "BETWEEN"
    IS_NULL = "IS_NULL"
    IS_NOT_NULL = "IS_NOT_NULL"


class SortDirection(str, Enum):
    ASC = "ASC"
    DESC = "DESC"


class SQLJoin(BaseModel):
    model_config = ConfigDict(extra="forbid")

    left_table: str
    right_table: str
    left_column: str
    right_column: str
    join_type: JoinType


class SQLFilter(BaseModel):
    model_config = ConfigDict(extra="forbid")

    column: str
    operator: FilterOperator
    values: list[str]

    @model_validator(mode="after")
    def validate_filter(self):

        null_operators = {
            FilterOperator.IS_NULL,
            FilterOperator.IS_NOT_NULL
        }

        scalar_operators = {
            FilterOperator.EQ,
            FilterOperator.NE,
            FilterOperator.GT,
            FilterOperator.GTE,
            FilterOperator.LT,
            FilterOperator.LTE
        }

        if self.operator in null_operators:
            if self.values:
                raise ValueError(
                    "NULL filters must not contain values."
                )

        elif self.operator == FilterOperator.BETWEEN:
            if len(self.values) != 2:
                raise ValueError(
                    "BETWEEN requires exactly two values."
                )

        elif self.operator in scalar_operators:
            if len(self.values) != 1:
                raise ValueError(
                    "Scalar filters require exactly one value."
                )

        elif not self.values:
            raise ValueError(
                "IN and NOT_IN require at least one value."
            )

        return self


class SQLOrder(BaseModel):
    model_config = ConfigDict(extra="forbid")

    field: str
    direction: SortDirection


class SQLTimePeriod(BaseModel):
    model_config = ConfigDict(extra="forbid")

    column: str
    start_date: str
    end_date: str


class SQLPlan(BaseModel):
    model_config = ConfigDict(extra="forbid")

    objective: str

    # Business meaning
    metric: str | None
    entity: str | None

    # Database requirements
    required_tables: list[str]
    joins: list[SQLJoin]
    filters: list[SQLFilter]

    # Result structure
    dimensions: list[str]
    group_by: list[str]
    order_by: list[SQLOrder]
    limit: int | None

    # Date restriction if required
    time_period: SQLTimePeriod | None

    # Any interpretation introduced during planning
    assumptions: list[str]

    @model_validator(mode="after")
    def validate_plan(self):

        if not self.required_tables:
            raise ValueError(
                "SQLPlan requires at least one table."
            )

        if self.limit is not None and self.limit < 1:
            raise ValueError(
                "SQLPlan limit must be greater than zero."
            )

        return self

In [3]:
example_plan = SQLPlan(
    objective="Count all completed orders.",

    metric="order_count",
    entity="orders",

    required_tables=[
        "orders"
    ],

    joins=[],

    filters=[
        SQLFilter(
            column="orders.order_status",
            operator=FilterOperator.EQ,
            values=["completed"]
        )
    ],

    dimensions=[],
    group_by=[],
    order_by=[],

    limit=None,
    time_period=None,

    assumptions=[]
)


print(
    example_plan.model_dump(
        mode="json"
    )
)

{'objective': 'Count all completed orders.', 'metric': 'order_count', 'entity': 'orders', 'required_tables': ['orders'], 'joins': [], 'filters': [{'column': 'orders.order_status', 'operator': 'EQ', 'values': ['completed']}], 'dimensions': [], 'group_by': [], 'order_by': [], 'limit': None, 'time_period': None, 'assumptions': []}


## Section 2 - LLM SQL Planner

This section connects the approved Day 3 question analysis to the structured SQL plan.

Only questions classified as answerable are allowed to reach the planner. The planner uses the approved interpretation, live database schema and business glossary to describe the intended query without generating SQL yet.|

In [4]:
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

In [19]:
# Find the project root
current_path = Path.cwd()

if current_path.name == "notebooks":
    PROJECT_ROOT = current_path.parent
else:
    PROJECT_ROOT = current_path


# Make the src package importable
src_path = PROJECT_ROOT / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


# Load private environment variables
load_dotenv(
    PROJECT_ROOT / ".env",
    override=True
)


# Reuse tested project code from previous days
from ai_analytics_assistant.database import (
    get_db_connection,
    get_schema_context,
)

from ai_analytics_assistant.question_analyzer import (
    QuestionStatus,
    analyze_question as reusable_analyze_question,
)


# Load the live schema
schema_context = get_schema_context()


# Load documented business definitions
with open(
    PROJECT_ROOT / "config" / "business_glossary.json",
    "r",
) as file:
    business_glossary = json.load(file)


SQL_PLANNER_VERSION = "sql_planner_v1"


client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    timeout=20.0,
    max_retries=2,
)


print("Planner version:", SQL_PLANNER_VERSION)
print("Schema loaded:", bool(schema_context))
print("Business metrics:", len(business_glossary))

Planner version: sql_planner_v1
Schema loaded: True
Business metrics: 8


In [6]:
def create_sql_plan(
    question: str,
    analysis,
) -> SQLPlan:

    # SQL planning is only allowed after Day 3 approval
    if analysis.status != QuestionStatus.ANSWERABLE:
        raise ValueError(
            "SQL planning is only allowed for "
            "ANSWERABLE questions."
        )

    instructions = f"""
You are the SQL planning layer of an AI analytics assistant.

Your task is to convert an APPROVED business-question analysis into
a structured SQL plan.

Do NOT generate SQL.

The plan will later be passed to a separate SQL generator and
deterministic SQL validator.


PLANNING RULES

1. APPROVED ANALYSIS

Treat the supplied question analysis as the approved business
interpretation.

Do not reclassify the question.

Do not introduce a new interpretation that conflicts with the
approved analysis.


2. DATABASE SCHEMA

Use only tables and columns that exist in the supplied PostgreSQL
schema.

Do not invent:
- tables
- columns
- relationships
- status values


3. BUSINESS METRICS

Use the documented business glossary when planning metrics.

If the approved metric is net_revenue, order_count,
average_order_value or another documented metric, preserve its
documented business meaning.


4. TABLES

required_tables must contain every table required to answer the
question and no unrelated tables.


5. JOINS

Create joins only when multiple tables are required.

Every join must correspond to a real relationship available in
the supplied schema.

Prefer the smallest set of joins required to answer the question.


6. FILTERS

Represent non-date restrictions using structured filters.

Examples include:
- completed order status
- store
- category
- product
- region


7. TIME PERIODS

When the approved analysis contains a concrete date range,
represent it using SQLTimePeriod.

Use the actual database date column that should enforce the period.

Do not duplicate the same date restriction inside filters.


8. GROUPING

Use dimensions and group_by only when the requested answer requires
results split by an entity or category.

Do not add grouping to questions asking for a single aggregate.


9. ORDERING

Use order_by only when ordering is required by the question.

For ranking requests, preserve the approved ranking direction.


10. LIMITS

Use the approved/default result limit when a ranking request has one.

Do not add a limit to a single aggregate result unless it is needed.


11. ASSUMPTIONS

Do not introduce unnecessary assumptions.

If the approved analysis already resolved a documented default,
preserve that interpretation rather than recording it again as a new
planning assumption.


12. SAFETY

This planner is strictly read-only.

Never create a plan for:
- INSERT
- UPDATE
- DELETE
- DROP
- ALTER
- CREATE
- TRUNCATE
- database administration

The Day 3 analyzer should already have rejected these requests.


DATABASE SCHEMA

{schema_context}


BUSINESS GLOSSARY

{json.dumps(business_glossary, indent=2)}
""".strip()


    approved_context = {
        "user_question": question,
        "approved_analysis": analysis.model_dump(
            mode="json"
        ),
    }


    response = client.responses.parse(
        model=os.getenv("OPENAI_MODEL"),
        instructions=instructions,
        input=json.dumps(
            approved_context,
            indent=2,
        ),
        text_format=SQLPlan,
        store=False,
    )


    if response.output_parsed is None:
        raise ValueError(
            "SQL planner returned no parsed output."
        )


    return response.output_parsed

In [7]:
planner_question = (
    "How many completed orders do we have?"
)

planner_analysis = reusable_analyze_question(
    planner_question
)


print(
    "Analysis status:",
    planner_analysis.status.value
)


planner_plan = create_sql_plan(
    planner_question,
    planner_analysis,
)


print(
    json.dumps(
        planner_plan.model_dump(mode="json"),
        indent=2,
    )
)

Analysis status: ANSWERABLE
{
  "objective": "Count completed customer orders.",
  "metric": "order_count",
  "entity": "orders",
  "required_tables": [
    "orders"
  ],
  "joins": [],
  "filters": [
    {
      "column": "orders.order_status",
      "operator": "EQ",
      "values": [
        "completed"
      ]
    }
  ],
  "dimensions": [],
  "group_by": [],
  "order_by": [],
  "limit": null,
  "time_period": null,
  "assumptions": []
}


## Section 3 - SQL Generation from Approved Plans

This section converts an approved structured SQL plan into PostgreSQL.

The generator receives the already-approved plan rather than interpreting the user's question again. This keeps business reasoning separate from SQL syntax and gives the deterministic safety layer a clear SQL statement to inspect next.

In [8]:
class GeneratedSQL(BaseModel):
    model_config = ConfigDict(extra="forbid")

    sql: str

In [9]:
SQL_GENERATOR_VERSION = "sql_generator_v1"


def generate_sql(
    plan: SQLPlan,
) -> GeneratedSQL:

    instructions = f"""
You are the PostgreSQL generation layer of an AI analytics assistant.

Your only task is to translate an APPROVED structured SQL plan into
one PostgreSQL read-only query.

Do not reinterpret the original business question.

Follow the structured plan exactly.


GENERATION RULES

1. POSTGRESQL

Generate PostgreSQL syntax only.


2. READ-ONLY QUERY

Generate exactly one read-only query.

Allowed top-level query forms:
- SELECT
- WITH ... SELECT

Never generate:
- INSERT
- UPDATE
- DELETE
- DROP
- ALTER
- CREATE
- TRUNCATE
- MERGE
- CALL
- COPY
- transaction commands
- administrative commands


3. APPROVED PLAN

Use the supplied structured SQL plan as the source of truth.

Do not:
- add new business assumptions
- change the requested metric
- change the requested filters
- change the requested time period
- introduce unrelated tables


4. DATABASE SCHEMA

Use only tables and columns that exist in the supplied schema.

Do not invent schema objects.


5. BUSINESS METRICS

Follow the documented business glossary exactly.

For documented metrics such as:
- net_revenue
- gross_sales
- discount_amount
- refund_amount
- order_count
- average_order_value

preserve their documented definitions.


6. JOINS

Use only joins required by the approved plan.

Join using real relationships in the database schema.


7. FILTERS

Translate the structured plan filters directly into PostgreSQL
predicates.

Preserve documented values exactly, including lowercase status values
such as 'completed' and 'cancelled'.


8. TIME PERIODS

Use the approved start and end dates exactly.

For DATE columns, use an inclusive date range unless the structured
plan specifies otherwise.


9. AGGREGATION

Use aggregation, GROUP BY and ordering only when required by the plan.

For order_count, preserve the documented meaning of distinct completed
orders.


10. RESULT LIMIT

Apply LIMIT only when the approved plan contains a limit.


11. OUTPUT

Return only the structured GeneratedSQL response.

The sql field must contain exactly one PostgreSQL query.

Do not include:
- Markdown fences
- explanations
- comments
- alternative queries


DATABASE SCHEMA

{schema_context}


BUSINESS GLOSSARY

{json.dumps(business_glossary, indent=2)}
""".strip()


    response = client.responses.parse(
        model=os.getenv("OPENAI_MODEL"),
        instructions=instructions,
        input=json.dumps(
            plan.model_dump(mode="json"),
            indent=2,
        ),
        text_format=GeneratedSQL,
        store=False,
    )


    if response.output_parsed is None:
        raise ValueError(
            "SQL generator returned no parsed output."
        )


    return response.output_parsed

In [10]:
generated_sql = generate_sql(
    planner_plan
)


print(
    "Generator version:",
    SQL_GENERATOR_VERSION
)

print()

print(
    generated_sql.sql
)

Generator version: sql_generator_v1

SELECT COUNT(DISTINCT orders.order_id) AS order_count
FROM orders
WHERE orders.order_status = 'completed';


## Section 4 - SQLGlot AST Validation

This section parses the generated PostgreSQL into an abstract syntax tree and applies deterministic safety rules before execution.

The validator allows only a single read-only SELECT query and rejects write operations, DDL, administrative commands and multiple statements. SQLGlot helps inspect the query structure, but it is only one layer of the final database safety controls.

In [11]:
import sqlglot
from sqlglot import exp

In [12]:
class SQLValidationResult(BaseModel):
    model_config = ConfigDict(extra="forbid")

    is_valid: bool
    statement_type: str | None
    tables: list[str]
    columns: list[str]
    errors: list[str]

In [28]:
FORBIDDEN_SQL_NODES = (
    exp.Insert,
    exp.Update,
    exp.Delete,
    exp.Merge,
    exp.Copy,
    exp.Create,
    exp.Drop,
    exp.Alter,
    exp.TruncateTable,
    exp.Into,
    exp.Command,
    exp.Transaction,
    exp.Commit,
    exp.Rollback,
    exp.Lock,
)

ALLOWED_TABLES = {
    "customers",
    "stores",
    "categories",
    "products",
    "orders",
    "order_items",
    "payments",
    "returns",
    "promotions",
}


def validate_sql(sql: str) -> SQLValidationResult:
    errors = []

    try:
        statements = [
            statement
            for statement in sqlglot.parse(
                sql,
                dialect="postgres",
            )
            if statement is not None
        ]

    except sqlglot.errors.ParseError as exc:
        return SQLValidationResult(
            is_valid=False,
            statement_type=None,
            tables=[],
            columns=[],
            errors=[
                f"SQL parsing failed: {exc}"
            ],
        )


    # Exactly one statement is allowed
    if len(statements) != 1:
        return SQLValidationResult(
            is_valid=False,
            statement_type=None,
            tables=[],
            columns=[],
            errors=[
                "Exactly one SQL statement is allowed."
            ],
        )


    tree = statements[0]

    statement_type = type(tree).__name__


    # Only SELECT / WITH ... SELECT queries are allowed
    if not isinstance(tree, exp.Select):
        errors.append(
            "Only SELECT queries are allowed."
        )


    # Reject dangerous operations anywhere in the AST
    for forbidden_type in FORBIDDEN_SQL_NODES:
        if tree.find(forbidden_type):
            errors.append(
                f"Forbidden SQL operation detected: "
                f"{forbidden_type.__name__}."
            )


    # Collect CTE names so they are not mistaken for database tables
    cte_names = {
        cte.alias_or_name
        for cte in tree.find_all(exp.CTE)
    }


    database_tables = []


    # Validate every real database table
    for table in tree.find_all(exp.Table):
        table_name = table.name
        schema_name = table.db


        # CTE references are allowed
        if (
            table_name in cte_names
            and not schema_name
        ):
            continue


        database_tables.append(
            table_name
        )


        # V1 only permits the controlled public schema
        if (
            schema_name
            and schema_name != "public"
        ):
            errors.append(
                f"Schema is not allowed: "
                f"{schema_name}."
            )


        # Only the ecommerce tables are permitted
        if table_name not in ALLOWED_TABLES:
            errors.append(
                f"Table is not allowed: "
                f"{table_name}."
            )


    tables = sorted(
        set(database_tables)
    )


    columns = sorted(
        {
            column.sql(
                dialect="postgres"
            )
            for column in tree.find_all(exp.Column)
        }
    )


    return SQLValidationResult(
        is_valid=not errors,
        statement_type=statement_type,
        tables=tables,
        columns=columns,
        errors=errors,
    )

In [14]:
validation = validate_sql(
    generated_sql.sql
)


print(
    validation.model_dump(
        mode="json"
    )
)

{'is_valid': True, 'statement_type': 'Select', 'tables': ['orders'], 'columns': ['orders.order_id', 'orders.order_status'], 'errors': []}


### Adversarial SQL Validation

The validator is now tested against normal queries and deliberately unsafe SQL.

These cases check whether deterministic AST rules can reject writes, destructive commands, hidden data-modifying CTEs, multiple statements and malformed SQL before anything reaches PostgreSQL.

In [16]:
adversarial_sql_cases = [
    {
        "case_id": "SAFE_001",
        "sql": (
            "SELECT COUNT(*) "
            "FROM orders "
            "WHERE order_status = 'completed';"
        ),
        "expected_valid": True,
    },
    {
        "case_id": "WRITE_001",
        "sql": "DELETE FROM customers;",
        "expected_valid": False,
    },
    {
        "case_id": "WRITE_002",
        "sql": (
            "UPDATE products "
            "SET list_price = 0;"
        ),
        "expected_valid": False,
    },
    {
        "case_id": "DDL_001",
        "sql": "DROP TABLE orders;",
        "expected_valid": False,
    },
    {
        "case_id": "MULTI_001",
        "sql": (
            "SELECT * FROM orders; "
            "DROP TABLE orders;"
        ),
        "expected_valid": False,
    },
    {
        "case_id": "CTE_WRITE_001",
        "sql": (
            "WITH deleted AS ("
            "DELETE FROM orders "
            "RETURNING order_id"
            ") "
            "SELECT * FROM deleted;"
        ),
        "expected_valid": False,
    },
    {
        "case_id": "SELECT_INTO_001",
        "sql": (
            "SELECT * "
            "INTO order_backup "
            "FROM orders;"
        ),
        "expected_valid": False,
    },
    {
        "case_id": "INVALID_001",
        "sql": "SELECT FROM WHERE;",
        "expected_valid": False,
    },
]


for case in adversarial_sql_cases:
    result = validate_sql(
        case["sql"]
    )

    passed = (
        result.is_valid
        == case["expected_valid"]
    )

    print(
        f"{case['case_id']} | "
        f"expected={case['expected_valid']} | "
        f"actual={result.is_valid} | "
        f"passed={passed}"
    )

    if result.errors:
        print(
            "Errors:",
            result.errors
        )

    print()

SAFE_001 | expected=True | actual=True | passed=True

WRITE_001 | expected=False | actual=False | passed=True
Errors: ['Only SELECT queries are allowed.', 'Forbidden SQL operation detected: Delete.']

WRITE_002 | expected=False | actual=False | passed=True
Errors: ['Only SELECT queries are allowed.', 'Forbidden SQL operation detected: Update.']

DDL_001 | expected=False | actual=False | passed=True
Errors: ['Only SELECT queries are allowed.', 'Forbidden SQL operation detected: Drop.']

MULTI_001 | expected=False | actual=False | passed=True
Errors: ['Exactly one SQL statement is allowed.']

CTE_WRITE_001 | expected=False | actual=False | passed=True
Errors: ['Forbidden SQL operation detected: Delete.']

SELECT_INTO_001 | expected=False | actual=False | passed=True
Errors: ['Forbidden SQL operation detected: Into.']

INVALID_001 | expected=False | actual=False | passed=True
Errors: ['SQL parsing failed: Expected table name but got <Token token_type: TokenType.WHERE, text: WHERE, line: 1

In [18]:
allowlist_cases = [
    {
        "case_id": "SAFE_CTE_001",
        "sql": (
            "WITH recent_orders AS ("
            "SELECT order_id FROM orders"
            ") "
            "SELECT COUNT(*) FROM recent_orders;"
        ),
        "expected_valid": True,
    },
    {
        "case_id": "SYSTEM_TABLE_001",
        "sql": (
            "SELECT * "
            "FROM pg_catalog.pg_user;"
        ),
        "expected_valid": False,
    },
    {
        "case_id": "UNKNOWN_TABLE_001",
        "sql": (
            "SELECT * "
            "FROM secret_table;"
        ),
        "expected_valid": False,
    },
]


for case in allowlist_cases:
    result = validate_sql(
        case["sql"]
    )

    passed = (
        result.is_valid
        == case["expected_valid"]
    )

    print(
        f"{case['case_id']} | "
        f"expected={case['expected_valid']} | "
        f"actual={result.is_valid} | "
        f"passed={passed}"
    )

    if result.errors:
        print(
            "Errors:",
            result.errors
        )

    print()

SAFE_CTE_001 | expected=True | actual=True | passed=True

SYSTEM_TABLE_001 | expected=False | actual=False | passed=True
Errors: ['Schema is not allowed: pg_catalog.', 'Table is not allowed: pg_user.']

UNKNOWN_TABLE_001 | expected=False | actual=False | passed=True
Errors: ['Table is not allowed: secret_table.']



## Section 5 - PostgreSQL Preflight and Resource Controls

This section sends validated SQL to PostgreSQL for a non-executing preflight check before the real query is allowed to run.

The database connection uses a read-only transaction together with statement and lock timeouts. PostgreSQL `EXPLAIN` is then used to confirm that the query can be planned successfully without executing the actual business query.

In [20]:
class SQLPreflightResult(BaseModel):
    model_config = ConfigDict(extra="forbid")

    passed: bool
    plan_lines: list[str]
    error: str | None

In [23]:
PREFLIGHT_STATEMENT_TIMEOUT_MS = 5000
PREFLIGHT_LOCK_TIMEOUT_MS = 2000


def preflight_sql(
    sql: str,
) -> SQLPreflightResult:

    validation = validate_sql(sql)

    if not validation.is_valid:
        return SQLPreflightResult(
            passed=False,
            plan_lines=[],
            error=(
                "SQL failed deterministic validation: "
                + "; ".join(validation.errors)
            ),
        )

    try:
        with get_db_connection() as connection:

            with connection.cursor() as cursor:

                # Keep the transaction explicitly read-only
                cursor.execute(
                    "SET TRANSACTION READ ONLY"
                )

                # Apply timeout settings only to this transaction
                cursor.execute(
                    """
                    SELECT set_config(
                        'statement_timeout',
                        %s,
                        true
                    )
                    """,
                    (
                        f"{PREFLIGHT_STATEMENT_TIMEOUT_MS}ms",
                    ),
                )

                cursor.execute(
                    """
                    SELECT set_config(
                        'lock_timeout',
                        %s,
                        true
                    )
                    """,
                    (
                        f"{PREFLIGHT_LOCK_TIMEOUT_MS}ms",
                    ),
                )

                # EXPLAIN plans the query without executing it
                cursor.execute(
                    "EXPLAIN " + sql
                )

                rows = cursor.fetchall()

                plan_lines = [
                    row[0]
                    for row in rows
                ]

        return SQLPreflightResult(
            passed=True,
            plan_lines=plan_lines,
            error=None,
        )

    except Exception as exc:
        return SQLPreflightResult(
            passed=False,
            plan_lines=[],
            error=str(exc),
        )

In [24]:
preflight = preflight_sql(
    generated_sql.sql
)


print(
    "Preflight passed:",
    preflight.passed
)

print(
    "Error:",
    preflight.error
)

print()

for line in preflight.plan_lines:
    print(line)

Preflight passed: True
Error: None

Aggregate  (cost=966.39..966.40 rows=1 width=8)
  ->  Index Scan using orders_pkey on orders  (cost=0.29..908.79 rows=23042 width=4)
        Filter: ((order_status)::text = 'completed'::text)


## Section 6 - Controlled Read-Only Execution

This section executes SQL only after deterministic validation and PostgreSQL preflight have succeeded.

The query runs inside a read-only transaction with execution timeouts and a result-row cap. This provides a final database-level safety layer even after the SQL has already passed the earlier checks.

In [25]:
class SQLExecutionResult(BaseModel):
    model_config = ConfigDict(extra="forbid")

    success: bool
    columns: list[str]
    rows: list[list]
    rows_returned: int
    result_truncated: bool
    error: str | None

In [26]:
EXECUTION_STATEMENT_TIMEOUT_MS = 5000
EXECUTION_LOCK_TIMEOUT_MS = 2000
MAX_RESULT_ROWS = 200


def execute_read_only_sql(
    sql: str,
) -> SQLExecutionResult:

    # Never execute SQL that fails deterministic validation
    validation = validate_sql(sql)

    if not validation.is_valid:
        return SQLExecutionResult(
            success=False,
            columns=[],
            rows=[],
            rows_returned=0,
            result_truncated=False,
            error=(
                "SQL failed deterministic validation: "
                + "; ".join(validation.errors)
            ),
        )


    # Require successful PostgreSQL planning first
    preflight = preflight_sql(sql)

    if not preflight.passed:
        return SQLExecutionResult(
            success=False,
            columns=[],
            rows=[],
            rows_returned=0,
            result_truncated=False,
            error=(
                "SQL failed PostgreSQL preflight: "
                + str(preflight.error)
            ),
        )


    try:
        with get_db_connection() as connection:

            with connection.cursor() as cursor:

                # Final database-level read-only protection
                cursor.execute(
                    "SET TRANSACTION READ ONLY"
                )

                # Limit execution time for this transaction
                cursor.execute(
                    """
                    SELECT set_config(
                        'statement_timeout',
                        %s,
                        true
                    )
                    """,
                    (
                        f"{EXECUTION_STATEMENT_TIMEOUT_MS}ms",
                    ),
                )

                cursor.execute(
                    """
                    SELECT set_config(
                        'lock_timeout',
                        %s,
                        true
                    )
                    """,
                    (
                        f"{EXECUTION_LOCK_TIMEOUT_MS}ms",
                    ),
                )

                # Execute the validated query
                cursor.execute(sql)


                columns = [
                    column.name
                    for column in cursor.description
                ]


                # Fetch one extra row to detect truncation
                fetched_rows = cursor.fetchmany(
                    MAX_RESULT_ROWS + 1
                )


                result_truncated = (
                    len(fetched_rows)
                    > MAX_RESULT_ROWS
                )


                rows = [
                    list(row)
                    for row in fetched_rows[
                        :MAX_RESULT_ROWS
                    ]
                ]


        return SQLExecutionResult(
            success=True,
            columns=columns,
            rows=rows,
            rows_returned=len(rows),
            result_truncated=result_truncated,
            error=None,
        )


    except Exception as exc:
        return SQLExecutionResult(
            success=False,
            columns=[],
            rows=[],
            rows_returned=0,
            result_truncated=False,
            error=str(exc),
        )

In [27]:
execution = execute_read_only_sql(
    generated_sql.sql
)

print(
    "Execution success:",
    execution.success
)

print(
    "Columns:",
    execution.columns
)

print(
    "Rows returned:",
    execution.rows_returned
)

print(
    "Truncated:",
    execution.result_truncated
)

print(
    "Error:",
    execution.error
)

print(
    "Rows:",
    execution.rows
)

Execution success: True
Columns: ['order_count']
Rows returned: 1
Truncated: False
Error: None
Rows: [[23042]]


### Adversarial Execution Tests

This section tests the complete execution boundary rather than only normal queries.

The tests verify that unsafe SQL is rejected before execution, locking reads are blocked, long-running queries are stopped by the timeout, and large result sets are capped before being returned to the application.

In [29]:
execution_safety_cases = [
    {
        "case_id": "UNSAFE_EXEC_001",
        "sql": "DELETE FROM customers;",
        "expected_success": False,
    },
    {
        "case_id": "LOCK_EXEC_001",
        "sql": (
            "SELECT * "
            "FROM orders "
            "LIMIT 1 "
            "FOR UPDATE;"
        ),
        "expected_success": False,
    },
    {
        "case_id": "TIMEOUT_001",
        "sql": "SELECT pg_sleep(10);",
        "expected_success": False,
    },
    {
        "case_id": "ROW_CAP_001",
        "sql": (
            "SELECT order_id "
            "FROM orders "
            "ORDER BY order_id;"
        ),
        "expected_success": True,
    },
]


for case in execution_safety_cases:
    result = execute_read_only_sql(
        case["sql"]
    )

    passed = (
        result.success
        == case["expected_success"]
    )

    print(
        f"{case['case_id']} | "
        f"expected={case['expected_success']} | "
        f"actual={result.success} | "
        f"passed={passed}"
    )

    print(
        f"rows={result.rows_returned} | "
        f"truncated={result.result_truncated}"
    )

    if result.error:
        print(
            "Error:",
            result.error
        )

    print()

UNSAFE_EXEC_001 | expected=False | actual=False | passed=True
rows=0 | truncated=False
Error: SQL failed deterministic validation: Only SELECT queries are allowed.; Forbidden SQL operation detected: Delete.

LOCK_EXEC_001 | expected=False | actual=False | passed=True
rows=0 | truncated=False
Error: SQL failed deterministic validation: Forbidden SQL operation detected: Lock.

TIMEOUT_001 | expected=False | actual=False | passed=True
rows=0 | truncated=False
Error: canceling statement due to statement timeout

ROW_CAP_001 | expected=True | actual=True | passed=True
rows=200 | truncated=True



## Section 7 - Reusable SQL Pipeline

This section moves the tested SQL planning, generation, validation and execution logic into reusable source modules.

Keeping these components outside the notebook allows the final application to use the same controlled SQL workflow without copying notebook code.

In [30]:
from ai_analytics_assistant.sql_planner import (
    create_sql_plan as reusable_create_sql_plan,
    generate_sql as reusable_generate_sql,
)

from ai_analytics_assistant.sql_safety import (
    validate_sql as reusable_validate_sql,
    preflight_sql as reusable_preflight_sql,
    execute_read_only_sql as reusable_execute_sql,
)

In [31]:
end_to_end_question = (
    "How many completed orders do we have?"
)


# Day 3: understand and approve the question
end_to_end_analysis = reusable_analyze_question(
    end_to_end_question
)


# Day 4: create the structured SQL plan
end_to_end_plan = reusable_create_sql_plan(
    end_to_end_question,
    end_to_end_analysis,
)


# Generate PostgreSQL from the approved plan
end_to_end_sql = reusable_generate_sql(
    end_to_end_plan
)


# Deterministic AST validation
end_to_end_validation = reusable_validate_sql(
    end_to_end_sql.sql
)


# PostgreSQL EXPLAIN preflight
end_to_end_preflight = reusable_preflight_sql(
    end_to_end_sql.sql
)


# Controlled read-only execution
end_to_end_execution = reusable_execute_sql(
    end_to_end_sql.sql
)


print(
    "Analysis:",
    end_to_end_analysis.status.value
)

print(
    "Plan metric:",
    end_to_end_plan.metric
)

print(
    "Generated SQL:",
    end_to_end_sql.sql
)

print(
    "Validation:",
    end_to_end_validation.is_valid
)

print(
    "Preflight:",
    end_to_end_preflight.passed
)

print(
    "Execution:",
    end_to_end_execution.success
)

print(
    "Result:",
    end_to_end_execution.rows
)

Analysis: ANSWERABLE
Plan metric: order_count
Generated SQL: SELECT COUNT(DISTINCT orders.order_id) AS order_count
FROM orders
WHERE orders.order_status = 'completed';
Validation: True
Preflight: True
Execution: True
Result: [[23042]]
